In [ ]:
import polars as pl
import os 
from pathlib import Path

In [ ]:
ROOT_DIR = "./DATA"

path=f"{ROOT_DIR}/StockUniteLegale_utf8.parquet"

In [ ]:
for file in os.listdir(ROOT_DIR):
   
    CURRENT_PATH = Path(f"{ROOT_DIR}/{file}")
    if CURRENT_PATH.suffix != ".parquet":
        continue
    df=pl.read_parquet(CURRENT_PATH,n_rows=1)
    print(f"File: {file}")
    print(df.columns)
    print(len(df.columns))
    print("\n")

File: prospecting_leads_2026.parquet
['siret', 'company_name', 'activitePrincipaleEtablissement', 'trancheEffectifsEtablissement', 'categorieEntreprise', 'address', 'libelleCommuneEtablissement', 'codePostalEtablissement', 'dateCreationUniteLegale', 'coordonneeLambertAbscisseEtablissement', 'coordonneeLambertOrdonneeEtablissement']
11


File: StockEtablissement_utf8.parquet
['siren', 'nic', 'siret', 'statutDiffusionEtablissement', 'dateCreationEtablissement', 'trancheEffectifsEtablissement', 'anneeEffectifsEtablissement', 'activitePrincipaleRegistreMetiersEtablissement', 'dateDernierTraitementEtablissement', 'etablissementSiege', 'nombrePeriodesEtablissement', 'complementAdresseEtablissement', 'numeroVoieEtablissement', 'indiceRepetitionEtablissement', 'dernierNumeroVoieEtablissement', 'indiceRepetitionDernierNumeroVoieEtablissement', 'typeVoieEtablissement', 'libelleVoieEtablissement', 'codePostalEtablissement', 'libelleCommuneEtablissement', 'libelleCommuneEtrangerEtablissement', 'di

In [ ]:
df = pl.read_parquet(path, n_rows=5)

In [ ]:
df.head()

siren,statutDiffusionUniteLegale,unitePurgeeUniteLegale,dateCreationUniteLegale,sigleUniteLegale,sexeUniteLegale,prenom1UniteLegale,prenom2UniteLegale,prenom3UniteLegale,prenom4UniteLegale,prenomUsuelUniteLegale,pseudonymeUniteLegale,identifiantAssociationUniteLegale,trancheEffectifsUniteLegale,anneeEffectifsUniteLegale,dateDernierTraitementUniteLegale,nombrePeriodesUniteLegale,categorieEntreprise,anneeCategorieEntreprise,dateDebut,etatAdministratifUniteLegale,nomUniteLegale,nomUsageUniteLegale,denominationUniteLegale,denominationUsuelle1UniteLegale,denominationUsuelle2UniteLegale,denominationUsuelle3UniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,nomenclatureActivitePrincipaleUniteLegale,nicSiegeUniteLegale,economieSocialeSolidaireUniteLegale,societeMissionUniteLegale,caractereEmployeurUniteLegale,activitePrincipaleNAF25UniteLegale
str,str,bool,date,str,str,str,str,str,str,str,str,str,str,i64,datetime[μs],i64,str,i64,date,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str
"""000325175""","""O""",null,2000-09-26,null,"""M""","""THIERRY""",null,null,null,"""THIERRY""",null,null,"""NN""",null,2025-12-06 10:43:55,6,"""PME""",2023,2018-02-07,"""A""","""JANOYER""",null,null,null,null,null,1000,"""32.12Z""","""NAFRev2""","""00065""",null,null,null,"""32.12Y"""
"""001807254""","""O""",null,1972-05-01,null,"""M""","""JACQUES-LUCIEN""",null,null,null,"""JACQUES-LUCIEN""",null,null,"""NN""",null,2024-03-22 14:26:06,5,null,null,2014-12-31,"""C""","""BRETON""",null,null,null,null,null,1000,"""85.59A""","""NAFRev2""","""00022""",null,null,null,null
"""005410220""","""O""",true,1954-12-25,null,"""M""","""GEORGES""",null,null,null,"""GEORGES""",null,null,"""NN""",null,2024-03-22 14:26:06,1,null,null,1988-03-31,"""C""","""WATTEBLED""",null,null,null,null,null,1000,"""22.02""","""NAP""","""00015""",null,null,null,null
"""005410345""","""O""",true,null,null,"""M""","""MICHEL""",null,null,null,"""MICHEL""",null,null,"""NN""",null,2024-03-22 14:26:06,1,null,null,1984-12-25,"""C""","""DEBRAY""",null,null,null,null,null,1000,"""79.06""","""NAP""","""00010""",null,null,null,null
"""005410394""","""O""",true,1954-12-25,null,"""M""","""ROBERT""","""ALFRED""",null,null,"""ROBERT""",null,null,"""NN""",null,2024-03-22 14:26:06,1,null,null,1987-12-01,"""C""","""DAULT""",null,null,null,null,null,1000,"""64.42""","""NAP""","""00018""",null,null,null,null


In [ ]:
import polars as pl
import os

# Optimized for 2026 hardware: Uses all CPU cores and Streaming API
os.environ["POLARS_MAX_THREADS"] = str(os.cpu_count())

def build_prospecting_database(etab_path, unite_path, output_path):
    """
    Optimized pipeline to merge 30M+ rows into a clean prospecting dataset.
    Uses Lazy API for query optimization and Streaming for memory safety.
    """
    
    # 1. Setup Lazy Scanners
    etab_scan = pl.scan_parquet(etab_path)
    unite_scan = pl.scan_parquet(unite_path)

    # 2. Filter Legal Units (Parent Data)
    # Target: Active companies that allow commercial diffusion
    unite_filtered = unite_scan.filter(
        (pl.col("statutDiffusionUniteLegale") == "O") & 
        (pl.col("etatAdministratifUniteLegale") == "A")
    ).select([
        "siren",
        "denominationUniteLegale",
        "nomUniteLegale",
        "prenom1UniteLegale",
        "categorieEntreprise",
        "dateCreationUniteLegale"
    ])

    # 3. Filter Establishments (Local Data)
    # Target: Active locations with employee counts
    etab_filtered = etab_scan.filter(
        (pl.col("statutDiffusionEtablissement") == "O") & 
        (pl.col("etatAdministratifEtablissement") == "A")
    ).select([
        "siren",
        "siret",
        "enseigne1Etablissement",
        "activitePrincipaleEtablissement",
        "trancheEffectifsEtablissement",
        "codePostalEtablissement",
        "libelleCommuneEtablissement",
        "numeroVoieEtablissement",
        "typeVoieEtablissement",
        "libelleVoieEtablissement",
        "coordonneeLambertAbscisseEtablissement",
        "coordonneeLambertOrdonneeEtablissement"
    ])

    # 4. Join and Feature Engineering
    # Parallel processing across all cores
    prospecting_pipeline = (
        etab_filtered.join(unite_filtered, on="siren", how="inner")
        .with_columns([
            # Priority: Company Name > Sign > Person Name
            pl.coalesce([
                pl.col("denominationUniteLegale"),
                pl.col("enseigne1Etablissement"),
                pl.format("{} {}", pl.col("prenom1UniteLegale"), pl.col("nomUniteLegale"))
            ]).alias("company_name"),
            
            # Clean unified address
            pl.format("{} {} {} {} {}", 
                pl.col("numeroVoieEtablissement").fill_null(""),
                pl.col("typeVoieEtablissement").fill_null(""),
                pl.col("libelleVoieEtablissement").fill_null(""),
                pl.col("codePostalEtablissement").fill_null(""),
                pl.col("libelleCommuneEtablissement").fill_null("")
            ).str.replace_all(r"\s+", " ").str.strip_chars().alias("address")
        ])
        .select([
            "siret",
            "company_name",
            "activitePrincipaleEtablissement",
            "trancheEffectifsEtablissement",
            "categorieEntreprise",
            "address",
            "libelleCommuneEtablissement",
            "codePostalEtablissement",
            "dateCreationUniteLegale",
            "coordonneeLambertAbscisseEtablissement",
            "coordonneeLambertOrdonneeEtablissement"
        ])
    )

    # 5. Execute via Streaming
    # Writes results to disk batch-by-batch to prevent RAM exhaustion
    print("Executing optimized multi-threaded merge...")
    prospecting_pipeline.sink_parquet(output_path)
    print(f"Extraction complete: {output_path}")



In [ ]:
if(os.path.exists("./DATA/prospecting_leads_2026.parquet")):
    print("Prospecting database already exists. Skipping extraction.")
else:
    build_prospecting_database(
            etab_path="./DATA/StockEtablissement_utf8.parquet",
            unite_path="./DATA/StockUniteLegale_utf8.parquet",
            output_path="./DATA/prospecting_leads_2026.parquet"
        )

Prospecting database already exists. Skipping extraction.


In [ ]:
DF8 = pl.read_parquet("./DATA/prospecting_leads_2026.parquet")

In [ ]:
DF8.unique().count()

siret,company_name,activitePrincipaleEtablissement,trancheEffectifsEtablissement,categorieEntreprise,address,libelleCommuneEtablissement,codePostalEtablissement,dateCreationUniteLegale,coordonneeLambertAbscisseEtablissement,coordonneeLambertOrdonneeEtablissement
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
14544249,14544237,14544249,14544249,8723929,14544249,14328425,14378060,14544249,11964087,11964087


In [ ]:
# Add this step to your pipeline to rename columns to meaningful English names
prospecting_pipeline = DF8.rename({
    "activitePrincipaleEtablissement": "industry_code",
    "trancheEffectifsEtablissement": "staff_size_range",
    "categorieEntreprise": "company_type",
    "libelleCommuneEtablissement": "city",
    "codePostalEtablissement": "zip_code",
    "dateCreationUniteLegale": "creation_date",
    "coordonneeLambertAbscisseEtablissement": "x_lambert_93",
    "coordonneeLambertOrdonneeEtablissement": "y_lambert_93"
})

In [ ]:
DF8 = (
    DF8
    .select([
        "siret",
        "company_name",
        "activitePrincipaleEtablissement",
        "trancheEffectifsEtablissement",
        "categorieEntreprise",
        "address",
        "libelleCommuneEtablissement",
        "codePostalEtablissement",
        "dateCreationUniteLegale",
        "coordonneeLambertAbscisseEtablissement",
        "coordonneeLambertOrdonneeEtablissement"
    ])
    .rename({
        "activitePrincipaleEtablissement": "industry_code",
        "trancheEffectifsEtablissement": "staff_size_range",
        "categorieEntreprise": "company_type",
        "libelleCommuneEtablissement": "city",
        "codePostalEtablissement": "zip_code",
        "dateCreationUniteLegale": "creation_date",
        "coordonneeLambertAbscisseEtablissement": "x_lambert_93",
        "coordonneeLambertOrdonneeEtablissement": "y_lambert_93"
    })
)

In [ ]:
DF8.head()

siret,company_name,industry_code,staff_size_range,company_type,address,city,zip_code,creation_date,x_lambert_93,y_lambert_93
str,str,str,str,str,str,str,str,date,str,str
"""29860058600010""","""ASSOCIATION FONCIERE DE MAILLE""","""42.99Z""","""NN""",null,"""4 RUE DE PICARD 86190 MAILLE""","""MAILLE""","""86190""",1961-06-20,"""477574.5788544328""","""6623602.835517963"""
"""29860060200015""","""ASS FONCIERE DE REMEMBREMENT D…","""42.99Z""","""NN""","""PME""","""MAIRIE 86330 MARTAIZE""","""MARTAIZE""","""86330""",1963-01-21,null,null
"""29860064400017""","""ASS FONCIERE DE REMEMB DE MAZE…","""42.99Z""","""NN""","""PME""","""MAIRIE 86110 MAZEUIL""","""MAZEUIL""","""86110""",1964-07-06,null,null
"""29860083400014""","""ASSOCIATION FONCIERE DE ROIFFE""","""42.99Z""","""NN""","""PME""","""PLACE D’AUMETZ 86120 ROIFFE""","""ROIFFE""","""86120""",1969-01-03,"""476678.2980433072""","""6673572.107521303"""
"""29860084200017""","""ASS FONCIERE DE REMEMBREMENT D…","""42.99Z""","""NN""","""PME""","""MAIRIE 86330 SAINT-CLAIR""","""SAINT-CLAIR""","""86330""",1967-06-06,null,null


In [ ]:
naf_mapping = pl.read_csv("./DATA/nomenclature-dactivites-francaise-naf-rev-2-code-ape.csv", separator=";").select([
    pl.col("code_naf").alias("industry_code"),
    pl.col("intitule_naf").alias("industry_description"),    
    pl.col("intitule_naf_65").alias("industry_group_65"),    
    pl.col("intitule_naf_40").alias("industry_sector_40") 
])

In [ ]:
naf_mapping.head(5)

industry_code,industry_description,industry_group_65,industry_sector_40
str,str,str,str
"""3011""","""Construction de navires et de …","""Construction de navires et de …","""Construct. navires & structure…"
"""4752A""","""Commerce de détail de quincail…","""Comm. détail de quincaillerie,…","""Com. dét. quinc. pein. etc. (m…"
"""1629Z""","""Fabrication d'objets divers en…","""Fabrication objets divers en b…","""Fab. objet div. bois, liège, v…"
"""6492Z""","""Autre distribution de crédit""","""Autre distribution de crédit""","""Autre distribution de crédit"""
"""8899""","""Autre action sociale sans hébe…","""Autre action sociale sans hébe…","""Autre action sociale sans hébr…"


In [ ]:
DF8=DF8.with_columns(
   pl.col("industry_code").str.replace(".", "", literal=True    )
)

In [ ]:
df = DF8.join(
    naf_mapping,
    on="industry_code",
    how="left"
)
df.head()

siret,company_name,industry_code,staff_size_range,company_type,address,city,zip_code,creation_date,x_lambert_93,y_lambert_93,industry_description,industry_group_65,industry_sector_40
str,str,str,str,str,str,str,str,date,str,str,str,str,str
"""29860058600010""","""ASSOCIATION FONCIERE DE MAILLE""","""4299Z""","""NN""",null,"""4 RUE DE PICARD 86190 MAILLE""","""MAILLE""","""86190""",1961-06-20,"""477574.5788544328""","""6623602.835517963""","""Construction d'autres ouvrages…","""Construction d'autres ouvrages…","""Constr. aut. ouvrage de génie …"
"""29860060200015""","""ASS FONCIERE DE REMEMBREMENT D…","""4299Z""","""NN""","""PME""","""MAIRIE 86330 MARTAIZE""","""MARTAIZE""","""86330""",1963-01-21,null,null,"""Construction d'autres ouvrages…","""Construction d'autres ouvrages…","""Constr. aut. ouvrage de génie …"
"""29860064400017""","""ASS FONCIERE DE REMEMB DE MAZE…","""4299Z""","""NN""","""PME""","""MAIRIE 86110 MAZEUIL""","""MAZEUIL""","""86110""",1964-07-06,null,null,"""Construction d'autres ouvrages…","""Construction d'autres ouvrages…","""Constr. aut. ouvrage de génie …"
"""29860083400014""","""ASSOCIATION FONCIERE DE ROIFFE""","""4299Z""","""NN""","""PME""","""PLACE D’AUMETZ 86120 ROIFFE""","""ROIFFE""","""86120""",1969-01-03,"""476678.2980433072""","""6673572.107521303""","""Construction d'autres ouvrages…","""Construction d'autres ouvrages…","""Constr. aut. ouvrage de génie …"
"""29860084200017""","""ASS FONCIERE DE REMEMBREMENT D…","""4299Z""","""NN""","""PME""","""MAIRIE 86330 SAINT-CLAIR""","""SAINT-CLAIR""","""86330""",1967-06-06,null,null,"""Construction d'autres ouvrages…","""Construction d'autres ouvrages…","""Constr. aut. ouvrage de génie …"


In [ ]:
df["industry_description"].unique()

industry_description
str
"""Activités des agences de trava…"
"""Édition de livres"""
"""Horlogerie"""
"""Construction de routes et auto…"
"""Activités d’ordre public et de…"
…
"""Commerce de détail d'enregistr…"
"""Gestion des musées"""
"""Réparation d'appareils électro…"


In [ ]:
code_postaux = pl.read_csv(
    "./DATA/base-officielle-des-codes-postaux.csv",
    separator=";",
    truncate_ragged_lines=True,
    ignore_errors=True,
    null_values=["2A001", "[ND]", "NA", ""]
)

In [ ]:
code_postaux.head()

Code_commune_INSEE,Nom_de_la_commune,Code_postal,Libellé_d_acheminement,Ligne_5,Departement,geom,centroid
i64,str,i64,str,str,i64,str,str
75104,"""PARIS 04""",75004,"""PARIS""",null,75,"""{""coordinates"": [[[[2.36849088…","""48.854350603, 2.357625608"""
75110,"""PARIS 10""",75010,"""PARIS""",null,75,"""{""coordinates"": [[[[2.36471357…","""48.876129334, 2.360715102"""
77051,"""BRAY SUR SEINE""",77480,"""BRAY SUR SEINE""",null,77,"""{""coordinates"": [[[[3.26229068…","""48.414017188, 3.242863551"""
77052,"""BREAU""",77720,"""BREAU""",null,77,"""{""coordinates"": [[[[2.89054255…","""48.557440136, 2.878536628"""
77055,"""BROU SUR CHANTEREINE""",77177,"""BROU SUR CHANTEREINE""",null,77,"""{""coordinates"": [[[[2.65745618…","""48.890767137, 2.639890249"""


In [ ]:
code_postaux.columns

['Code_commune_INSEE',
 'Nom_de_la_commune',
 'Code_postal',
 'Libellé_d_acheminement',
 'Ligne_5',
 'Departement',
 'geom',
 'centroid']

In [ ]:
code_postaux["Code_postal"].value_counts().sort("count", descending=True).head(10)

Code_postal,count
i64,u32
77320,24
77760,16
91150,15
95450,15
78270,14
77410,14
95420,14
77440,13
77560,13
